
# Define Scenario Parameters

The scenario analysis examines from three potential external changes standpoint (economic shifts, demographic changes, and environmental conditions). Each scenario adjusts selected dataset variables to evaluate their potential influence on heart disease classification and the model's ability to minimize false negatives.

### Scenario 1: Economic shifts - Increased Healthcare Costs

Increased healthcare costs could reduce access to preventive care and regular medical checkups.

Assumptions: Increase resting blood pressure (trestbps) and cholesterol by 10%, keeping other variables unchanged.

Rationale: This scenario examines whether potentially worsening cardiovascular health indicators influence the model's ability to identify patients with heart disease.

### Scenario 2: Demographic changes - Aging Population

An aging population could increase the representation of older adults in the patient population.

Assumptions: Increase patient age age by five years and decrease maximum heart rate (trestbps) by 5%, keeping other variables unchanged.

Rationale: This scenario examines whether changes in patient age and maximum heart rate influence heart disease classification outcomes.

### Scenario 3: Environmental conditions -Increased Air Pollution

Increased air pollution could introduce additional cardiovascular health challenges.

Assumptions: Increase resting blood pressure (trestbps) by 5%, decrease maximum heart rate (trestbps) by 5%, and increase the proportion of patients with exercise-induced angina (exang) by 10 percentage points.

Rationale: This scenario examines whether potential changes in cardiovascular health indicators influence the model's ability to identify patients with heart disease.


In [84]:
# Import required libraries
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
from datetime import datetime
# Define project information
todays_date = datetime.now().strftime("%Y%m%d")
student_id = "abrgeb8774"
# Display project information
display(Markdown("### Perform Scenario Analysis"))
display(Markdown(f"Date: {todays_date}"))
display(Markdown(f"Student ID: {student_id}"))

### Perform Scenario Analysis

Date: 20261108

Student ID: abrgeb8774

In [85]:
# Display project information
display(Markdown("### Perform Scenario Analysis"))
display(Markdown(f"Date: {todays_date}"))
display(Markdown(f"Student ID: {student_id}"))
# Generate baseline predictions
baseline_predictions = optimized_model.predict(X_test)
baseline_predictions = np.argmax(baseline_predictions, axis=1)
# Convert test data back to original feature values
X_test_original = pd.DataFrame(
    scaler.inverse_transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)
# Create scenario datasets
healthcare_costs_data = X_test_original.copy()
healthcare_costs_data["trestbps"] *= 1.10
healthcare_costs_data["chol"] *= 1.10
aging_population_data = X_test_original.copy()
aging_population_data["age"] += 5
aging_population_data["thalch"] *= 0.95
air_pollution_data = X_test_original.copy()
air_pollution_data["trestbps"] *= 1.05
air_pollution_data["thalch"] *= 0.95
false_indices = air_pollution_data[air_pollution_data["exang_False"] == 1].index
num_to_change = int(round(0.10 * len(air_pollution_data)))
selected_indices = false_indices[:num_to_change]
air_pollution_data.loc[selected_indices, "exang_False"] = 0
air_pollution_data.loc[selected_indices, "exang_True"] = 1
# Standardize scenario datasets
healthcare_costs_data = pd.DataFrame(
    scaler.transform(healthcare_costs_data),
    columns=X_test.columns,
    index=X_test.index
)
aging_population_data = pd.DataFrame(
    scaler.transform(aging_population_data),
    columns=X_test.columns,
    index=X_test.index
)
air_pollution_data = pd.DataFrame(
    scaler.transform(air_pollution_data),
    columns=X_test.columns,
    index=X_test.index
)
# Define scenario datasets and names
scenario_names = [
    "Increased Healthcare Costs",
    "Aging Population",
    "Increased Air Pollution"
]
scenario_datasets = [
    healthcare_costs_data,
    aging_population_data,
    air_pollution_data
]
# Perform scenario analysis
results = []
results.append({
    "Scenario": "Baseline",
    "Changed Predictions": 0,
    "Total Predictions": len(baseline_predictions)
})
for i in range(len(scenario_names)):
    scenario_predictions = optimized_model.predict(scenario_datasets[i])
    scenario_predictions = np.argmax(scenario_predictions, axis=1)
    changed_predictions = np.sum(scenario_predictions != baseline_predictions)
    results.append({
        "Scenario": scenario_names[i],
        "Changed Predictions": changed_predictions,
        "Total Predictions": len(scenario_predictions)
    })
# Convert results to DataFrame
results_df = pd.DataFrame(results)
# Display results
display(results_df)

### Perform Scenario Analysis

Date: 20261108

Student ID: abrgeb8774

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 


,Scenario,Changed Predictions,Total Predictions
0,Baseline,0,184
1,Increased Healthcare Costs,6,184
2,Aging Population,9,184
3,Increased Air Pollution,8,184


In [86]:
# Display project information
display(Markdown("### Perform Sensitivity and Impact Analysis"))
display(Markdown(f"Date: {todays_date}"))
display(Markdown(f"Student ID: {student_id}"))
# Generate baseline predictions
baseline_predictions = optimized_model.predict(X_test, verbose=0)
baseline_predictions = np.argmax(baseline_predictions, axis=1)
# Convert test data back to original feature values
X_test_original = pd.DataFrame(
    scaler.inverse_transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)
# Define sensitivity parameters
age_changes = [1, 3, 5]
thalch_changes = [0.01, 0.03, 0.05]
# Initialize results
sensitivity_results = []
# Perform sensitivity analysis for age
for age_change in age_changes:
    # Create a copy of the original test dataset
    sensitivity_data = X_test_original.copy()
    # Increase patient age
    sensitivity_data["age"] += age_change
    # Standardize the modified dataset
    sensitivity_data = pd.DataFrame(
        scaler.transform(sensitivity_data),
        columns=X_test.columns,
        index=X_test.index
    )
    # Generate predictions
    sensitivity_predictions = optimized_model.predict(
        sensitivity_data, verbose=0
    )
    sensitivity_predictions = np.argmax(
        sensitivity_predictions, axis=1
    )
    # Compare predictions with baseline
    changed_predictions = np.sum(
        sensitivity_predictions != baseline_predictions
    )
    # Store results
    sensitivity_results.append({
        "Feature": "Age",
        "Change": f"+{age_change} years",
        "Changed Predictions": changed_predictions,
        "Total Predictions": len(baseline_predictions)
    })
# Perform sensitivity analysis for maximum heart rate
for thalch_change in thalch_changes:
    # Create a copy of the original test dataset
    sensitivity_data = X_test_original.copy()
    # Decrease maximum heart rate
    sensitivity_data["thalch"] *= (1 - thalch_change)
    # Standardize the modified dataset
    sensitivity_data = pd.DataFrame(
        scaler.transform(sensitivity_data),
        columns=X_test.columns,
        index=X_test.index
    )
    # Generate predictions
    sensitivity_predictions = optimized_model.predict(
        sensitivity_data, verbose=0
    )
    sensitivity_predictions = np.argmax(
        sensitivity_predictions, axis=1
    )
    # Compare predictions with baseline
    changed_predictions = np.sum(
        sensitivity_predictions != baseline_predictions
    )
    # Store results
    sensitivity_results.append({
        "Feature": "Maximum Heart Rate",
        "Change": f"-{thalch_change * 100:.0f}%",
        "Changed Predictions": changed_predictions,
        "Total Predictions": len(baseline_predictions)
    })
# Convert results to DataFrame
sensitivity_results_df = pd.DataFrame(sensitivity_results)
# Display sensitivity analysis results
display(sensitivity_results_df)

### Perform Sensitivity and Impact Analysis

Date: 20261108

Student ID: abrgeb8774

,Feature,Change,Changed Predictions,Total Predictions
0,Age,+1 years,2,184
1,Age,+3 years,5,184
2,Age,+5 years,6,184
3,Maximum Heart Rate,-1%,0,184
4,Maximum Heart Rate,-3%,2,184
5,Maximum Heart Rate,-5%,6,184


### Sensitivity and Impact Analysis

The sensitivity analysis examined how changes in patient age and maximum heart rate influenced the heart disease classification predictions. The results showed that increasing patient age by five years and decreasing maximum heart rate by 5% each resulted in six changed predictions out of 184. Smaller changes produced fewer classification differences, indicating that the model's predictions were more sensitive to larger changes in these features. In real-world healthcare settings, these findings highlight the importance of considering patient age and maximum heart rate when assessing heart disease risk and supporting clinical decision-making.